# 03 — New model: ConvNeXt-Base

Default config: `convnext_legacy.yaml`, the recipe that actually produced the 92.77% run (gradual unfreezing, local normalization, RandAugment + Cutout, legacy dropout), re-measured over seeds. New recipes are new YAML files: change `CONFIG_PATH`.

Protocol (Phase 0): seeded runs, split check + fingerprint, checkpoint selected on validation, test reported with the val-selected weights and with the last epoch, binary metrics as in the paper. All hyperparameters live in the YAML config.

In [ ]:
import os, sys
from pathlib import Path

BRANCH = "main"
IS_COLAB = os.path.exists("/content")

if IS_COLAB:
    from google.colab import drive, userdata
    drive.mount("/content/drive")
    PROJECT_ROOT = Path("/content/Cloud_detection_project")
    if not PROJECT_ROOT.exists():
        token = userdata.get("GITHUB_TOKEN")
        os.system(f"git clone https://{token}@github.com/caMatt99/Cloud_detection_project.git {PROJECT_ROOT}")
else:
    PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

os.chdir(PROJECT_ROOT)
os.system(f"git fetch -q origin && git checkout -q {BRANCH} && git pull -q origin {BRANCH}")
os.system("git log -1 --format='commit %h  %s'")   # the commit this run uses

src = str(PROJECT_ROOT / "src")
if src not in sys.path:
    sys.path.insert(0, src)
print(f"✓ project root: {PROJECT_ROOT}  (colab={IS_COLAB})")
# NB: after a git pull that changes src/, restart the runtime — modules already imported are not reloaded.

In [ ]:
import wandb

if IS_COLAB:
    wandb.login(key=userdata.get("WANDB_API_KEY"))
else:
    wandb.login()  # ~/.netrc

In [ ]:
from utils import load_config
from dataset import check_split

CONFIG_PATH = "configs/convnext_legacy.yaml"
SEEDS = None          # None -> experiment.seeds from the config; e.g. [0] for a quick check

cfg = load_config(CONFIG_PATH)
print(f"experiment : {cfg['experiment']['name']}  seeds={SEEDS or cfg['experiment']['seeds']}")
print(f"dataset    : {cfg['dataset']['path']}")
print(f"split      : fingerprint {check_split(cfg['dataset']['path'])}")

## Train over seeds

One W&B run per seed, grouped by `experiment.name`.

In [ ]:
from experiment import run_seeds

table = run_seeds(cfg, SEEDS)

## Comparison with the paper

`accuracy` in %, the rest binary (positive = `true`) in [0, 1]. 1 test image = 0.21 pp.

In [ ]:
import pandas as pd
from evaluate import PAPER_RESNET50

mean, std = table.loc["mean"], table.loc["std"]
fmt = lambda tag, k: f"{mean[f'test_{tag}_{k}']:.4f} ± {std[f'test_{tag}_{k}']:.4f}"
pd.DataFrame({
    "paper ResNet50 (single run)": PAPER_RESNET50,
    "val-selected (mean ± std)":   {k: fmt("best", k) for k in PAPER_RESNET50},
    "last epoch (mean ± std)":     {k: fmt("last", k) for k in PAPER_RESNET50},
})